# Primera entrega proyecto de investigación
Grupo REST pAPIs

### Introducción
Este notebook presenta un análisis exploratorio del dataset de resultados de las pruebas Saber 11, con el objetivo de comprender su estructura, identificar variables relevantes y analizar relaciones preliminares entre el desempeño académico y factores socioeconómicos.

**Nota:** El tamaño del dataset superaba el límite que admite databricks, por eso tocó dividirlo en 2.

## Descripción de los datos

### Carga de librerías

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

### Carga de datos
En esta sección se cargan ambas tablas desde el esquema del proyecto y se unifican en un solo DataFrame para facilitar el análisis.

#### Lectura y unión de datos

In [0]:
df_part1 = spark.table("proyecto.default.icfes_part_1")
df_part2 = spark.table("proyecto.default.icfes_part_2")

df_icfes = df_part1.unionByName(df_part2)

print("Parte 1:", df_part1.count())
print("Parte 2:", df_part2.count())
print("Total:", df_icfes.count())

### Vista preliminar

Se visualizan algunos registros para verificar que la unión se haya realizado correctamente y familiarizarse con la estructura de los datos.

In [0]:
display(df_icfes.limit(10))

### Estructura y tipos de datos

Se inspeccionan las columnas y sus tipos de datos para identificar variables numéricas y categóricas relevantes para el análisis.

In [0]:
df_icfes.printSchema()

**Descripción general del contenido**

El dataset contiene información individual de estudiantes que presentaron la prueba Saber 11, incluyendo variables académicas, características de la institución educativa y variables socioeconómicas del hogar. Entre las columnas más relevantes para este análisis se encuentran:

PUNT_GLOBAL: puntaje global del estudiante

FAMI_TIENEINTERNET: acceso a internet en el hogar

FAMI_TIENECOMPUTADOR: disponibilidad de computador

COLE_AREA_UBICACION: ubicación del colegio (urbano/rural)

### Descripción detallada de los datos - ICFES

Este dataset contiene información académica, sociodemográfica y del entorno educativo de los estudiantes en Colombia.

Está compuesto por variables de tipo numérico, categórico y de fecha, organizadas en las siguientes dimensiones:

---

#### 1. Identificación del registro

Variables que permiten identificar de manera única cada estudiante y periodo:

- **PERIODO (long):** Año o periodo de aplicación del examen  
- **ESTU_TIPODOCUMENTO (string):** Tipo de documento del estudiante  
- **ESTU_CONSECUTIVO (string):** Identificador único del estudiante  

---

#### 2. Información de la institución educativa (COLE_*)

Describe características del colegio donde estudia el estudiante:

##### Ubicación
- **COLE_AREA_UBICACION:** Rural o urbano  
- **COLE_DEPTO_UBICACION / COLE_MCPIO_UBICACION:** Departamento y municipio  

##### Características institucionales
- **COLE_BILINGUE:** Indica si el colegio es bilingüe  
- **COLE_CALENDARIO:** Tipo de calendario (A/B)  
- **COLE_CARACTER:** Académico, técnico, entre otros  
- **COLE_GENERO:** Mixto, femenino o masculino  
- **COLE_JORNADA:** Mañana, tarde o nocturna  

##### Naturaleza y tipo
- **COLE_NATURALEZA:** Público o privado  

##### Identificadores
- **COLE_COD_DANE_ESTABLECIMIENTO**  
- **COLE_COD_DANE_SEDE**  
- **COLE_CODIGO_ICFES**  

##### Nombre
- **COLE_NOMBRE_ESTABLECIMIENTO**  
- **COLE_NOMBRE_SEDE**  
- **COLE_SEDE_PRINCIPAL**  

---

#### 3. Ubicación del estudiante (ESTU_*)

Información geográfica del estudiante:

##### Lugar de presentación
- **ESTU_DEPTO_PRESENTACION**  
- **ESTU_MCPIO_PRESENTACION**  

##### Lugar de residencia
- **ESTU_DEPTO_RESIDE**  
- **ESTU_MCPIO_RESIDE**  
- **ESTU_PAIS_RESIDE**  

##### Códigos DANE asociados
- **ESTU_COD_DEPTO_PRESENTACION**  
- **ESTU_COD_MCPIO_PRESENTACION**  
- **ESTU_COD_RESIDE_DEPTO**  
- **ESTU_COD_RESIDE_MCPIO**  

---

#### 4. Características demográficas del estudiante

- **ESTU_FECHANACIMIENTO (date):** Fecha de nacimiento  
- **ESTU_GENERO (string):** Género del estudiante  
- **ESTU_NACIONALIDAD (string):** Nacionalidad  
- **ESTU_PRIVADO_LIBERTAD (string):** Condición de privación de libertad  
- **ESTU_ESTUDIANTE (string):** Estado del estudiante  
- **ESTU_ESTADOINVESTIGACION (string):** Estado del registro  

---

#### 5. Condiciones socioeconómicas del hogar (FAMI_*)

##### Educación de los padres
- **FAMI_EDUCACIONMADRE**  
- **FAMI_EDUCACIONPADRE**  

##### Condiciones del hogar
- **FAMI_ESTRATOVIVIENDA**  
- **FAMI_PERSONASHOGAR**  
- **FAMI_CUARTOSHOGAR**  

##### Acceso a bienes
- **FAMI_TIENEAUTOMOVIL**  
- **FAMI_TIENELAVADORA**  
- **FAMI_TIENECOMPUTADOR**  
- **FAMI_TIENEINTERNET**  

---

#### 6. Resultados académicos

Variables de desempeño en la prueba Saber 11:

##### Puntajes por área
- **PUNT_MATEMATICAS**  
- **PUNT_LECTURA_CRITICA**  
- **PUNT_C_NATURALES**  
- **PUNT_SOCIALES_CIUDADANAS**  
- **PUNT_INGLES**  

##### Clasificación cualitativa
- **DESEMP_INGLES**  

##### Puntaje global
- **PUNT_GLOBAL**

### Número de columnas y nombres

In [0]:
print("Número de columnas:", len(df_icfes.columns))
df_icfes.columns

## Exploración de datos

#### 1. Estadísticos descriptivos del puntaje global
Se calculan estadísticas descriptivas del puntaje global para entender el comportamiento general del rendimiento académico.

In [0]:
df_icfes.select("PUNT_GLOBAL").describe().show()


#### 2. Distribución del puntaje global
Se analiza la distribución del puntaje global para identificar la concentración de estudiantes en diferentes niveles de desempeño.

In [0]:
df_icfes.select("PUNT_GLOBAL") \
    .dropna() \
    .sample(fraction=0.2, seed=42) \
    .toPandas()["PUNT_GLOBAL"].hist(bins=30)

#### 3. Promedio de puntaje por departamento
Se analiza el promedio del puntaje global por departamento, con el fin de identificar posibles diferencias territoriales en el desempeño académico.

In [0]:
df_icfes.groupBy("COLE_DEPTO_UBICACION") \
    .agg(
        F.count("*").alias("cantidad_estudiantes"),
        F.round(F.avg("PUNT_GLOBAL"), 2).alias("promedio_puntaje")
    ) \
    .orderBy(F.desc("promedio_puntaje")) \
    .show(10)

#### 4. Distribución por rangos de puntaje
Se agrupan los puntajes en rangos para facilitar la interpretación del desempeño académico.

In [0]:
df_icfes.groupBy(
    F.when(F.col("PUNT_GLOBAL") < 200, "<200")
     .when((F.col("PUNT_GLOBAL") >= 200) & (F.col("PUNT_GLOBAL") < 300), "200-300")
     .otherwise("300+").alias("rango_puntaje")
).count().show()

### 5. Frecuencia de acceso a internet
Se analiza la proporción de estudiantes con y sin acceso a internet en el hogar.

In [0]:
df_internet = df_icfes.groupBy("FAMI_TIENEINTERNET").count().toPandas()

ax = df_internet.plot(kind="bar", x="FAMI_TIENEINTERNET", y="count")

for idx, row in df_internet.iterrows():
    ax.annotate(f"{int(row['count'])}", (idx, row['count']), ha='center', va='bottom')

#### 6. Internet vs puntaje promedio
Se compara el puntaje promedio según el acceso a internet, identificando posibles diferencias en el rendimiento académico.

In [0]:
df_plot = df_icfes.groupBy("FAMI_TIENEINTERNET") \
    .agg(F.avg("PUNT_GLOBAL").alias("promedio_puntaje")) \
    .toPandas()

ax = df_plot.plot(kind="bar", x="FAMI_TIENEINTERNET", y="promedio_puntaje")

for idx, row in df_plot.iterrows():
    ax.annotate(f"{row['promedio_puntaje']:.2f}", (idx, row['promedio_puntaje']), ha='center', va='bottom')

#### 7. Distribución urbano vs rural
Se analiza la distribución de estudiantes según la ubicación del colegio.

In [0]:
df_icfes.groupBy("COLE_AREA_UBICACION").count().show()

#### 8. Promedio de puntajes por área
Se calculan los promedios de las distintas áreas evaluadas para identificar diferencias en el desempeño académico por componente.

In [0]:
display(
    df_icfes.select(
        F.round(F.avg(F.expr("try_cast(replace(PUNT_LECTURA_CRITICA, ',', '.') as double)")), 2).alias("avg_punt_lectura_critica"),
        F.round(F.avg(F.expr("try_cast(replace(PUNT_MATEMATICAS, ',', '.') as double)")), 2).alias("avg_punt_matematicas"),
        F.round(F.avg(F.expr("try_cast(replace(PUNT_C_NATURALES, ',', '.') as double)")), 2).alias("avg_punt_c_naturales"),
        F.round(F.avg(F.expr("try_cast(replace(PUNT_SOCIALES_CIUDADANAS, ',', '.') as double)")), 2).alias("avg_punt_sociales_ciudadanas"),
        F.round(F.avg(F.expr("try_cast(replace(PUNT_INGLES, ',', '.') as double)")), 2).alias("avg_punt_ingles")
    )
)

### Conclusiones de la exploración

A partir del análisis exploratorio del dataset de resultados Saber 11, se identificaron varios patrones relevantes en la estructura y comportamiento de los datos.

En primer lugar, los estadísticos descriptivos del puntaje global evidencian un promedio de **252.30**, con una desviación estándar de **50.43**, valores mínimos de **0** y máximos de **495**, lo que indica una amplia dispersión en el desempeño académico de los estudiantes.

Asimismo, la distribución del puntaje global muestra una concentración significativa en rangos intermedios, donde aproximadamente **2.9 millones** de estudiantes se ubican entre **200 y 300 puntos**, mientras que cerca de **700 mil** se encuentran por debajo de 200, evidenciando que la mayoría presenta desempeños medios.

En cuanto a las condiciones de acceso a tecnología, se observa que **3,733,676** estudiantes reportan tener acceso a internet en el hogar, frente a **3,184,404** que no cuentan con este servicio, lo que refleja una brecha digital aún significativa.

Adicionalmente, el análisis del puntaje promedio según acceso a internet indica que los estudiantes con conectividad presentan un promedio de **265.18**, frente a **234.83** en aquellos sin acceso, evidenciando una diferencia de más de **30 puntos**, lo que sugiere una fuerte relación entre acceso a internet y desempeño académico.

Por otra parte, la distribución de estudiantes según ubicación del colegio muestra una alta concentración en zonas urbanas, con más de **6.1 millones** de registros, frente a aproximadamente **1 millón** en zonas rurales, lo que evidencia una marcada desigualdad en la representación territorial.

El análisis por departamento permite identificar diferencias regionales significativas, donde Bogotá presenta los promedios más altos (hasta **274.78**), mientras otros departamentos se ubican en rangos cercanos a **253 puntos**, reflejando brechas territoriales en el desempeño académico.

Finalmente, los promedios por área muestran valores relativamente homogéneos entre componentes, con resultados cercanos a **50 puntos** en todas las áreas evaluadas, lo que sugiere un desempeño equilibrado entre las distintas competencias académicas.

## REPORTE DE CALIDAD DE DATOS
El reporte de calidad de datos tiene como objetivo identificar la presencia de valores faltantes en variables clave del dataset, con el fin de evaluar su impacto en el análisis y proponer estrategias de tratamiento adecuadas.

### Calidad de datos: valores nulos

Se revisan valores faltantes en variables clave para identificar posibles problemas de calidad de datos.

In [0]:

df_icfes.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_icfes.columns
]).show()

In [0]:
# Conteo de nulos en variables clave
df_icfes.select(
    F.count("*").alias("total_registros"),
    F.count("PUNT_GLOBAL").alias("no_nulos_punt_global"),
    (F.count("*") - F.count("PUNT_GLOBAL")).alias("nulos_punt_global"),
    F.count("FAMI_TIENEINTERNET").alias("no_nulos_internet"),
    (F.count("*") - F.count("FAMI_TIENEINTERNET")).alias("nulos_internet"),
    F.count("FAMI_TIENECOMPUTADOR").alias("no_nulos_computador"),
    (F.count("*") - F.count("FAMI_TIENECOMPUTADOR")).alias("nulos_computador")
).show()

### Reporte de calidad de datos

A partir del análisis de valores faltantes en el dataset de resultados Saber 11, se identificaron distintos niveles de completitud en las variables evaluadas.

El dataset cuenta con un total de **7,109,704 registros**, de los cuales **4,500,181** presentan valores válidos en el puntaje global, mientras que **2,609,523** registros corresponden a valores faltantes en esta variable. Esto representa aproximadamente un **36.7% de datos faltantes** en la variable más crítica del análisis, lo cual constituye una limitación importante.

Las variables relacionadas con acceso a tecnología presentan una alta completitud. En el caso de la variable de acceso a internet, se identifican **191,624 valores faltantes**, lo que equivale a cerca del **2.7% del total**, mientras que la variable de disponibilidad de computador presenta **151,217 valores faltantes**, equivalente a aproximadamente **2.1%**. Estos niveles pueden considerarse bajos en comparación con el tamaño total del dataset.

Estos resultados evidencian que la principal problemática de calidad de datos se concentra en la variable **PUNT_GLOBAL**, mientras que las variables socioeconómicas presentan una cobertura adecuada para el análisis.

### Propuesta de tratamiento de datos

Con base en los hallazgos, se proponen las siguientes estrategias:

- **Eliminación de registros con puntaje nulo:**  
  Dado que el puntaje global es la variable objetivo del análisis, se recomienda excluir los **2.6 millones de registros** sin esta información, ya que no aportan valor analítico y podrían distorsionar los resultados.

- **Imputación para variables socioeconómicas:**  
  Para variables como acceso a internet y computador, dado su bajo nivel de valores faltantes, se puede aplicar imputación mediante la categoría más frecuente (moda), sin afectar significativamente la distribución de los datos.

- **Tratamiento como categoría independiente:**  
  Alternativamente, los valores faltantes en variables categóricas pueden ser tratados como una categoría adicional (por ejemplo, "Sin información"), permitiendo conservar todos los registros.

- **Uso de subconjuntos filtrados:**  
  Para análisis específicos, se recomienda trabajar con subconjuntos de datos completos en las variables de interés, asegurando mayor consistencia en los resultados.

## Filtros, limpieza y transformación inicial

### Eliminación de registros sin puntaje global

Dado que el puntaje global es la variable objetivo del análisis, se eliminan los registros que no contienen esta información, ya que no aportan valor analítico y pueden generar sesgos en los resultados.

In [0]:
df_icfes_clean = df_icfes.filter(F.col("PUNT_GLOBAL").isNotNull())

print("Total original:", df_icfes.count())
print("Total limpio:", df_icfes_clean.count())

### Estandarización de variables categóricas

Se realiza una limpieza básica de variables categóricas relacionadas con acceso a tecnología, con el fin de asegurar consistencia en los valores (por ejemplo, evitar variaciones como "Si", "sí", "SI").

In [0]:
df_icfes_clean = df_icfes_clean.withColumn(
    "FAMI_TIENEINTERNET",
    F.upper(F.col("FAMI_TIENEINTERNET"))
)

df_icfes_clean = df_icfes_clean.withColumn(
    "FAMI_TIENECOMPUTADOR",
    F.upper(F.col("FAMI_TIENECOMPUTADOR"))
)

### Tratamiento inicial de valores faltantes

Para variables categóricas como acceso a internet y computador, se propone un tratamiento inicial de los valores faltantes mediante la asignación de la categoría "SIN INFORMACION", permitiendo conservar los registros sin introducir supuestos fuertes.

In [0]:
df_icfes_clean = df_icfes_clean.fillna({
    "FAMI_TIENEINTERNET": "SIN INFORMACION",
    "FAMI_TIENECOMPUTADOR": "SIN INFORMACION"
})

### Creación de variable binaria de acceso a internet

Se genera una variable binaria que indica si el estudiante tiene acceso a internet, facilitando su uso en análisis posteriores y modelos.

In [0]:
df_icfes_clean = df_icfes_clean.withColumn(
    "TIENE_INTERNET_BIN",
    F.when(F.col("FAMI_TIENEINTERNET") == "SI", 1).otherwise(0)
)
df_icfes_clean.select("TIENE_INTERNET_BIN").show(20)


### Filtrado de valores atípicos

Se realiza un filtrado básico eliminando valores extremos no válidos en el puntaje global (por ejemplo, valores iguales a 0), los cuales pueden corresponder a registros incompletos o inconsistentes.

In [0]:
df_icfes_clean = df_icfes_clean.filter(F.col("PUNT_GLOBAL") > 0)

### Estandarización de nombres de departamento

Se realiza una normalización básica de los nombres de departamento para evitar inconsistencias (por ejemplo, diferencias en acentos como "BOGOTA" y "BOGOTÁ"), lo cual facilita análisis territoriales posteriores.

In [0]:
df_icfes_clean = df_icfes_clean.withColumn(
    "COLE_DEPTO_UBICACION",
    F.upper(F.col("COLE_DEPTO_UBICACION"))
)

df_icfes_clean = df_icfes_clean.withColumn(
    "COLE_DEPTO_UBICACION",
    F.regexp_replace("COLE_DEPTO_UBICACION", "Á", "A")
)

### Creación de variable categórica de rangos de puntaje

Se construye una variable categórica que agrupa el puntaje global en rangos, facilitando el análisis del desempeño académico por niveles.

In [0]:
df_icfes_clean = df_icfes_clean.withColumn(
    "RANGO_PUNT_GLOBAL",
    F.when(F.col("PUNT_GLOBAL") < 200, "BAJO")
     .when((F.col("PUNT_GLOBAL") >= 200) & (F.col("PUNT_GLOBAL") < 300), "MEDIO")
     .otherwise("ALTO")
)

### Filtrado de registros sin información de ubicación

Para ciertos análisis territoriales, se excluyen registros sin información de ubicación del colegio, con el fin de garantizar consistencia en los resultados.

In [0]:
df_icfes_geo = df_icfes_clean.filter(
    F.col("COLE_AREA_UBICACION").isNotNull()
)

En esta fase se realizaron procesos iniciales de filtrado, limpieza y transformación de los datos con el fin de mejorar su calidad y consistencia. Se eliminaron registros sin puntaje global, se estandarizaron variables categóricas, se trataron valores faltantes mediante imputación básica y se generaron variables derivadas como indicadores binarios y rangos de puntaje. Estas acciones permiten contar con un dataset más confiable y preparado para análisis posteriores, dejando planteadas bases sólidas para un tratamiento más avanzado en la siguiente etapa del proyecto.